In [ ]:
import os
import warnings

os.environ["USE_PYGEOS"] = "0"
warnings.filterwarnings("ignore")

import shutil

# single cell and spatial packages
import anndata
import scanpy as sc
import spatialdata as sd
import dask.array as da # not strictly singlecell/spatial but we'll use it to access certain parts of spatialdata object

# plotting
import seaborn as sns
import matplotlib.pyplot as plt
import spatialdata_plot
from napari_spatialdata import Interactive
from spatialdata import bounding_box_query

# I have my datasets in the same dir as my code under a subdir called datasets
working_dir = os.getcwd()

### Visium: spot-level whole-transcriptome data

In [ ]:
# zarr is a chunked storage format for multidimensional arrays great for partial IO and parallelism
# zarr stores are just directories that you can inspect
visium_path = os.path.join(working_dir, "datasets/visium/data.zarr")
os.listdir(visium_path)

In [ ]:
# read zarr as spatialdata object
visium_sdata = sd.read_zarr(visium_path)
visium_sdata

In [ ]:
# sdata['table'] is the anndata object you're familiar with from day 1
visium_sdata["table"]

In [ ]:
# spot data looks slightly different
visium_sdata["table"].obs

In [ ]:
# some information about our features
visium_sdata['table'].var

In [ ]:
# check out the spot x feature table
visium_sdata["table"].to_df()

In [ ]:
# view just the visium H&E
visium_sdata.pl.render_images().pl.show("CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres")

In [ ]:
# view the visium H&E with spots overlaid
visium_sdata.pl.render_images().pl.render_shapes().pl.show("CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres")

In [ ]:
# Find a highly expressed gene to overlay
visium_sdata["table"].to_df().sum(axis=0).sort_values(ascending=False).head(10)

In [ ]:
# plot gene expression per spot over H&E
(
    visium_sdata.pl.render_images(elements="CytAssist_FFPE_Human_Breast_Cancer_lowres_image")
    .pl.render_shapes(elements="CytAssist_FFPE_Human_Breast_Cancer", color="COL1A1")
    .pl.show(coordinate_systems="CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres")
)

In [ ]:
# even better, we can explore these datasets interactively using napari!
# napari can also be used to manually annotate regions and subset spatialdata object by
interactive = Interactive(visium_sdata)
interactive.run()

In [ ]:
# identifty mitochondrial genes and calculate QC metrics with scanpy
visium_sdata['table'].var['mt'] = visium_sdata['table'].var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(visium_sdata['table'], qc_vars=["mt"], inplace=True, log1p=True)

# results added to obs and var
visium_sdata['table']

In [ ]:
# the number of genes with at least one count in a cell
# total counts across all genes per cell
# percent of counts that map to mitochondrial genes per cell

# using 15% here, generally higher than scRNA but visium spots contain multiple cells 
# and we don't want to lose specific tissues that might be more metabollically active #FIXME: citation needed 

# compare to 10x web summary stats for Visium FFPE breast cancer

sc.pl.violin(
    visium_sdata['table'],
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# we expect high gene counts coming from high cell density areas and tumor areas
fig, axs = plt.subplots(ncols=3, nrows=1, figsize=(12, 3))

metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']

for i,m in enumerate(metrics):

    (
        visium_sdata.pl.render_images(elements="CytAssist_FFPE_Human_Breast_Cancer_lowres_image")
        .pl.render_shapes(elements="CytAssist_FFPE_Human_Breast_Cancer", color=m)
        .pl.show(
            coordinate_systems="CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres",
            ax=axs[i], 
            title=m)
    )

plt.tight_layout()

In [ ]:
# filter out spot with fewer than 100 unique genes and filter out genes expressed in fewer than 3 spots
# this is quite lenient
sc.pp.filter_cells(visium_sdata['table'], min_genes=100)
sc.pp.filter_genes(visium_sdata['table'], min_cells=3)

# Normalizing to median total counts
sc.pp.normalize_total(visium_sdata['table'])
# Logarithmize the data
sc.pp.log1p(visium_sdata['table'])

# feature selection
sc.pp.highly_variable_genes(visium_sdata['table'], n_top_genes=2000)
# sc.pl.highly_variable_genes(visium_sdata['table'])

# PCA
sc.tl.pca(visium_sdata['table'])
# sc.pl.pca_variance_ratio(visium_sdata['table'], n_pcs=50, log=True)

# feature space neighbors and UMAP 
sc.pp.neighbors(visium_sdata['table'])
sc.tl.umap(visium_sdata['table'])

# leiden clustering
sc.tl.leiden(visium_sdata['table'], flavor='igraph', n_iterations=2)
sc.pl.umap(visium_sdata['table'], color='leiden')

In [ ]:
# plot spot cluster assignments over H&E
(
    visium_sdata
    .pl.render_shapes(elements="CytAssist_FFPE_Human_Breast_Cancer", color="leiden")
    .pl.show(coordinate_systems="CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres")
)

In [ ]:
# Find genes that are differentially expressed by Leiden cluster
sc.tl.rank_genes_groups(visium_sdata['table'], groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(visium_sdata['table'], groupby="leiden", standard_scale="var", n_genes=5)

In [ ]:
# CDH2 is a marker for invasive breast cancer and is enriched in clusters 5, 6, 14, 16
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(12, 5))

(
    visium_sdata
    .pl.render_shapes(elements="CytAssist_FFPE_Human_Breast_Cancer", color="CDH2")
    .pl.show(
        coordinate_systems="CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres",
        ax=axs[0],
        title='CDH2 Expression'
    )
)

(
    visium_sdata
    .pl.render_shapes(elements="CytAssist_FFPE_Human_Breast_Cancer", color="leiden", groups=['5', '6', '14', '16'])
    .pl.show(
        coordinate_systems="CytAssist_FFPE_Human_Breast_Cancer_downscaled_lowres",
        ax=axs[1],
        title='Candidate invasive DCIS clusters'
    )
)

plt.tight_layout()

The above are spots, which are collections of cells. To infer single-cell composition, we would have to 
do spot deconvolution. There are several tools for this. One example using cell2location can be seen here for this dataset: 

https://github.com/scverse/spatialdata-notebooks/blob/main/notebooks/paper_reproducibility/03_annotate_visium_cell2location.ipynb

### Xenium single-cell targeted spatial transcriptomics

In [ ]:
# xenium and scRNA reference inputs
xenium_path = os.path.join(working_dir, "datasets/xenium/data.zarr")
atlas_path = os.path.join(working_dir, "datasets/sc_atlas/BC_atlas_xe.h5ad")

# read xenium as spatialdata
xenium_sdata = sd.read_zarr(xenium_path)

# read single cell atlas as anndata object
bc_sc_atlas_adata = sc.read(atlas_path)

In [ ]:
# inspect, note the difference in the number of observations (cells now) and features (limited panel)
xenium_sdata

In [ ]:
# function for getting a cropped view of the entire image
def crop_to_bbox(x):
    return bounding_box_query(
        x,
        min_coordinate=[0, 10000],
        max_coordinate=[2000, 11000],
        axes=("x", "y"),
        target_coordinate_system="global",
    )

# overlay the segmentation on the morphology image
crop_to_bbox(xenium_sdata).pl.render_images(
    "morphology_focus"
).pl.render_shapes(
        "cell_boundaries", 
        fill_alpha=0, 
        outline_width=0.3, 
        outline_color='red'
        ).pl.show(
            coordinate_systems="global", 
            colorbar=False, 
            title='Segmentation overlay on DAPI'
        )


In [ ]:
# overlay the segmentation on the morphology image
crop_to_bbox(xenium_sdata).pl.render_images(
    "morphology_focus"
).pl.render_shapes(
        "cell_boundaries", 
        fill_alpha=0, 
        outline_width=0.3, 
        outline_color='red'
        ).pl.render_points(
            "transcripts",
            color='feature_name',
            groups='EPCAM',
            palette='orange'
        ).pl.show(
            coordinate_systems="global", 
            colorbar=False, 
            title='Segmentation overlay on DAPI'
        )

In [ ]:
# each transcript has a quality score. The transcripts matrix keeps all, but counts matrix is filtered to Q>20
# important note: This is not a sequencing quality score
# it quantifies the likelihood of seeing the observed fluorescence codeword
transcript_stats = xenium_sdata['transcripts']['qv'].describe()
transcript_stats_results = da.compute(transcript_stats)
[round(x) for x in transcript_stats_results]

In [ ]:
# create a hard copy of the feature table that we'll use below for cell type mapping from scRNA
adata = xenium_sdata['table'].copy()

cprobes = (
    adata.obs["control_probe_counts"].sum() / adata.obs["total_counts"].sum() * 100
)
cwords = (
    adata.obs["control_codeword_counts"].sum() / adata.obs["total_counts"].sum() * 100
)

# probes that are designed specifically not to bind to anything in the tissue
# used to detect non-specific or off-target binding
print(f"Negative DNA probe count % : {cprobes}")

# fluorescent probe codes that don't map to one of the target genes
# used to detect errors in decoding
print(f"Negative decoding count % : {cwords}")

In [ ]:
# use scanpy to calculate some QC metrics and plot
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

fig, axs = plt.subplots(1, 4, figsize=(15, 4))

# metrics similar to scRNA-seq
axs[0].set_title("Total transcripts per cell")
sns.histplot(
    adata.obs["total_counts"],
    kde=False,
    ax=axs[0],
)

axs[1].set_title("Unique transcripts per cell")
sns.histplot(
    adata.obs["n_genes_by_counts"],
    kde=False,
    ax=axs[1],
)

# metrics more similar to imaging modalities
axs[2].set_title("Area of segmented cells")
sns.histplot(
    adata.obs["cell_area"],
    kde=False,
    ax=axs[2],
)

axs[3].set_title("Nucleus ratio")
sns.histplot(
    adata.obs["nucleus_area"] / adata.obs["cell_area"],
    kde=False,
    ax=axs[3],
)

In [ ]:
# map major cell types (publication does an extra step to get high-res cell types but we don't have time for that)
# take the intersection of genes between reference atlas and xenium dataset
genes = list(set(bc_sc_atlas_adata.var_names) & set(adata.var_names))

# subset to intersected genes
bc_sc_atlas_adata = bc_sc_atlas_adata[:, genes]
adata = adata[:, genes]

# normalize and log transform counts
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# map xenium data to reference sing ingest
sc.pp.pca(bc_sc_atlas_adata)
sc.pp.neighbors(bc_sc_atlas_adata)
sc.tl.umap(bc_sc_atlas_adata)
sc.tl.ingest(adata, bc_sc_atlas_adata, obs="celltype_major")

In [ ]:
# Find genes that are differentially expressed by cell type
sc.tl.rank_genes_groups(adata, groupby="celltype_major", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, groupby="celltype_major", standard_scale="var", n_genes=5)

In [ ]:
# link the table to the cell boundaries so we can color boundaries by cell type
xenium_sdata.tables["table"].obs["region"] = "cell_boundaries"
xenium_sdata.set_table_annotates_spatialelement(
    table_name="table", 
    region="cell_boundaries", 
    instance_key="cell_id"
)

# bring the cell type annotations back into our original feature table in the sdata object
xenium_sdata['table'].obs['celltype_major'] = list(adata.obs['celltype_major'])

# plot cancer epithelial cells and EpCAM puncta as validation
crop_to_bbox(xenium_sdata).pl.render_images(
    "morphology_focus"
).pl.render_shapes(
        "cell_boundaries", 
        color='celltype_major',
        groups='Cancer Epithelial',
        palette='red'
        ).pl.render_points(
            "transcripts",
            color='feature_name',
            groups='EPCAM',
            palette='orange'
        ).pl.show(
            coordinate_systems="global", 
            colorbar=False, 
            title='Segmentation overlay on DAPI'
        )

Now that we have cell types, there's a lot of different directions that we could go.
Squidpy could now be used to build a spatial neighborhood graph and calculate spatial statistics 
We could also use the normalized counts and look at gene expression gradients or gene set enrichment analysis